# Training dan Perbandingan Model Prediksi Hotspot (APIKAH)

Notebook ini melatih dan membandingkan beberapa model untuk memprediksi **kemunculan hotspot 1–3 hari ke depan** dari kondisi iklim dan riwayat hotspot.

## Data

- Panel: `dataset_apikah/variabel_iklim/*.parquet`
- 19.301.621 baris, 3.583 sel grid 0,25°, 5.387 hari (2012-01-01 – 2026-09-30)
- Label: `ada_api` (0/1) per sel-hari
- Target utama: `y3 = 1` jika ada hotspot di sel yang sama pada t+1, t+2, atau t+3

## Pembagian waktu

| Split | Periode | Perkiraan baris | Positif |
|---|---|---|---|
| Train | 2012-01-01 – 2022-12-31 | 14,40 juta | 8,63% |
| Validasi | 2023-01-01 – 2023-12-31 | 1,31 juta | 9,88% |
| Uji | 2024-01-01 – 2026-09-30 | 3,60 juta | 8,25% |

Pembagian kronologis, bukan acak, untuk mencegah kebocoran waktu.

## Model yang dibandingkan

1. Persistensi: prediksi `y3=1` jika hari ini ada hotspot.
2. Klimatologi: laju historis per sel × bulan.
3. Regresi logistik.
4. Random Forest.
5. LightGBM.
6. XGBoost.
7. MLP neural network (PyTorch, fallback scikit-learn).

Metrik utama: **PR-AUC**. ROC-AUC hanya pelengkap karena kelas positif hanya sekitar 8–9%.

## Cara menjalankan

```bash
pip install duckdb pyarrow pandas numpy scikit-learn lightgbm joblib matplotlib seaborn
# opsional, paket xgboost bisa besar karena dependensi CUDA:
pip install xgboost
# opsional untuk MLP PyTorch:
pip install torch --index-url https://download.pytorch.org/whl/cpu
```

Untuk uji cepat tanpa melatih penuh, set environment variable `SMOKE_TEST=1`. Notebook akan memakai sampel kecil dan iterasi pendek.

Perkiraan kebutuhan RAM untuk training penuh: 8 GB. Jika memori terbatas, turunkan `MAX_NEG` (misalnya 1.000.000) atau `NEG_PER_POS` menjadi 1.

Semua artefak ditulis ke `modeling/artifacts/` (di-ignore Git).

## Step 0: Setup, dependency, dan konfigurasi

In [ ]:
from pathlib import Path
import os
import json
import time
import warnings

import duckdb
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    brier_score_loss,
    log_loss,
)
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False


def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "dataset_apikah").exists() and (p / "modeling").exists():
            return p
    raise FileNotFoundError("Root repo APIKAH tidak ditemukan.")


ROOT = find_root()
IKLIM_DIR = ROOT / "dataset_apikah" / "variabel_iklim"
if not IKLIM_DIR.exists():
    IKLIM_DIR = ROOT / "dataset_apikah" / "training_iklim" / "variabel_iklim"

ARTIFACT_DIR = ROOT / "modeling" / "artifacts"
MODEL_DIR = ARTIFACT_DIR / "models"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(ARTIFACT_DIR / "duckdb_tmp").mkdir(parents=True, exist_ok=True)

PANEL_PATH = ARTIFACT_DIR / "model_panel.parquet"
METRICS_PATH = ARTIFACT_DIR / "model_metrics.csv"
THRESHOLD_PATH = ARTIFACT_DIR / "model_thresholds.json"

FORCE_REBUILD = os.environ.get("FORCE_REBUILD", "0") == "1"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
SEED = 42

TRAIN_END = "2022-12-31"
VAL_START = "2023-01-01"
VAL_END = "2023-12-31"
TEST_START = "2024-01-01"

NEG_PER_POS = 2
MAX_NEG = 200_000 if SMOKE_TEST else 2_500_000

con = duckdb.connect()
con.execute(f"SET threads TO {max(1, min(8, os.cpu_count() or 2))}")
con.execute("SET preserve_insertion_order=false")
con.execute(f"SET temp_directory='{ARTIFACT_DIR / 'duckdb_tmp'}'")
con.execute("SET max_memory='4GB'")


def q(sql):
    return con.execute(sql).fetchdf()


print("Root        :", ROOT)
print("Panel       :", PANEL_PATH)
print("LightGBM    :", HAS_LGB)
print("XGBoost     :", HAS_XGB)
print("PyTorch     :", HAS_TORCH)
print("SMOKE_TEST  :", SMOKE_TEST)

## Step 1: Bangun panel fitur dan target

Panel dibangun sekali dari Parquet mentah, lalu di-cache. Set `FORCE_REBUILD=1` untuk membangun ulang.

Fitur yang dibuat:
- Lag hotspot: `lag1`, `lag2`, `lag3`, `lag7`, `lag14`, `lag30`
- Akumulasi hotspot: `hotspot_7d`, `hotspot_30d`
- Target: `lead1`, `lead2`, `lead3`, `y3`

Semua fitur hanya memakai informasi sampai hari t.

In [ ]:
if PANEL_PATH.exists() and not FORCE_REBUILD:
    print("Cache ditemukan, lewati pembangunan panel:", PANEL_PATH)
else:
    t0 = time.time()
    files = f"read_parquet('{IKLIM_DIR / '*.parquet'}')"
    sql = f"""
    WITH base AS (
        SELECT
            date::DATE AS date,
            lat_c, lon_c,
            bulan, hari_dalam_tahun,
            t2m_max, rh_min, vpd_max, wind_max, swvl1_mean,
            precip_mm, fwi, dc, kbdi, hujan_7h, hujan_30h,
            nino34_lag1, dmi_lag1,
            ada_api, n_hotspot
        FROM {files}
    ),
    w AS (
        SELECT *,
            lag(ada_api, 1)  OVER win AS lag1,
            lag(ada_api, 2)  OVER win AS lag2,
            lag(ada_api, 3)  OVER win AS lag3,
            lag(ada_api, 7)  OVER win AS lag7,
            lag(ada_api, 14) OVER win AS lag14,
            lag(ada_api, 30) OVER win AS lag30,
            sum(ada_api) OVER (
                PARTITION BY lat_c, lon_c ORDER BY date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) AS hotspot_7d,
            sum(ada_api) OVER (
                PARTITION BY lat_c, lon_c ORDER BY date
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
            ) AS hotspot_30d,
            lead(ada_api, 1) OVER win AS lead1,
            lead(ada_api, 2) OVER win AS lead2,
            lead(ada_api, 3) OVER win AS lead3
        FROM base
        WINDOW win AS (PARTITION BY lat_c, lon_c ORDER BY date)
    )
    SELECT *,
        CASE
            WHEN lead1 IS NULL OR lead2 IS NULL OR lead3 IS NULL THEN NULL
            ELSE greatest(lead1, lead2, lead3)
        END AS y3
    FROM w
    """
    con.execute(f"COPY ({sql}) TO '{PANEL_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    print(f"Panel dibangun dalam {time.time() - t0:.1f} detik")
    print("Ukuran (MB):", round(PANEL_PATH.stat().st_size / 1e6, 1))

print(q(f"SELECT count(*) AS baris, min(date) AS awal, max(date) AS akhir, round(100.0*avg(y3),2) AS positif_pct FROM read_parquet('{PANEL_PATH}')"))

## Step 2: Audit panel dan pembagian waktu

In [ ]:
P = f"read_parquet('{PANEL_PATH}')"

split_stats = q(f"""
SELECT 'train' AS split, count(*) AS baris, round(100.0*avg(y3),2) AS positif_pct
FROM {P} WHERE date <= DATE '{TRAIN_END}'
UNION ALL
SELECT 'validasi', count(*), round(100.0*avg(y3),2)
FROM {P} WHERE date BETWEEN DATE '{VAL_START}' AND DATE '{VAL_END}'
UNION ALL
SELECT 'uji', count(*), round(100.0*avg(y3),2)
FROM {P} WHERE date >= DATE '{TEST_START}'
""")
display(split_stats)

display(q(f"""
SELECT year(date) AS tahun,
       count(*) AS baris,
       sum(y3) AS positif,
       round(100.0*avg(y3),2) AS positif_pct
FROM {P} GROUP BY 1 ORDER BY 1
"""))

## Step 3: Sampel training

Kelas positif hanya sekitar 8,6%. Untuk mempercepat dan menyeimbangkan, training memakai **semua positif** ditambah sampel negatif sekitar 2× jumlah positif. Validasi dan uji tetap memakai data penuh.

Rasio ini tidak mengubah urutan risiko, tetapi memengaruhi kalibrasi probabilitas. Kalibrasi tetap perlu dilakukan sebelum probabilitas dipakai sebagai tingkat risiko.

In [ ]:
n_pos = q(f"SELECT count(*) FROM {P} WHERE date <= DATE '{TRAIN_END}' AND y3 = 1").iloc[0, 0]
n_neg_target = int(min(n_pos * NEG_PER_POS, MAX_NEG))
print("Positif training:", f"{n_pos:,}")
print("Target negatif  :", f"{n_neg_target:,}")

con.execute(f"""
CREATE OR REPLACE TABLE train_pos AS
SELECT * FROM {P}
WHERE date <= DATE '{TRAIN_END}' AND y3 = 1
""")

con.execute(f"""
CREATE OR REPLACE TABLE train_neg AS
SELECT * FROM (
    SELECT * FROM {P}
    WHERE date <= DATE '{TRAIN_END}' AND y3 = 0
) USING SAMPLE {n_neg_target} ROWS
""")

train = con.execute("""
SELECT * FROM train_pos
UNION ALL
SELECT * FROM train_neg
""").fetchdf()

val = q(f"""
SELECT * FROM {P}
WHERE date BETWEEN DATE '{VAL_START}' AND DATE '{VAL_END}' AND y3 IS NOT NULL
""")
test = q(f"""
SELECT * FROM {P}
WHERE date >= DATE '{TEST_START}' AND y3 IS NOT NULL
""")

if SMOKE_TEST:
    train = train.sample(n=min(60_000, len(train)), random_state=SEED).reset_index(drop=True)
    val = val.sample(n=min(30_000, len(val)), random_state=SEED).reset_index(drop=True)
    test = test.sample(n=min(30_000, len(test)), random_state=SEED).reset_index(drop=True)

for df in (train, val, test):
    for c in df.columns:
        if df[c].dtype == "float64":
            df[c] = df[c].astype("float32")
    df["bulan_sin"] = np.sin(2 * np.pi * df["bulan"] / 12).astype("float32")
    df["bulan_cos"] = np.cos(2 * np.pi * df["bulan"] / 12).astype("float32")
    df["doy_sin"] = np.sin(2 * np.pi * df["hari_dalam_tahun"] / 365.25).astype("float32")
    df["doy_cos"] = np.cos(2 * np.pi * df["hari_dalam_tahun"] / 365.25).astype("float32")

print()
print("Train :", train.shape, f"positif {100*train['y3'].mean():.2f}%")
print("Val   :", val.shape, f"positif {100*val['y3'].mean():.2f}%")
print("Test  :", test.shape, f"positif {100*test['y3'].mean():.2f}%")

## Step 4: Definisi fitur

`ada_api`, `n_hotspot`, `lead1..3`, dan `y3` adalah target atau label. Semua itu tidak boleh masuk fitur.

In [ ]:
FEATURES = [
    "lat_c", "lon_c",
    "bulan", "hari_dalam_tahun", "bulan_sin", "bulan_cos", "doy_sin", "doy_cos",
    "t2m_max", "rh_min", "vpd_max", "wind_max", "swvl1_mean",
    "precip_mm", "fwi", "dc", "kbdi", "hujan_7h", "hujan_30h",
    "nino34_lag1", "dmi_lag1",
    "lag1", "lag2", "lag3", "lag7", "lag14", "lag30",
    "hotspot_7d", "hotspot_30d",
]
TARGET = "y3"
SKEWED = ["precip_mm", "fwi", "dc", "kbdi", "hujan_7h", "hujan_30h"]

print("Jumlah fitur:", len(FEATURES))
print(FEATURES)

## Step 5: Helper evaluasi

Ambang dipilih pada validasi dengan F1 maksimum, lalu dipakai apa adanya pada uji. Ini mencegah optimasi ambang pada data uji.

In [ ]:
def best_threshold(y_true, prob):
    precision, recall, thresholds = precision_recall_curve(y_true, prob)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    idx = int(np.nanargmax(f1[:-1]))
    return float(thresholds[idx])


def evaluate(name, split, y_true, prob, threshold=None):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    if threshold is None:
        threshold = best_threshold(y_true, prob)
    pred = (prob >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, average="binary", zero_division=0
    )
    return {
        "model": name,
        "split": split,
        "n": len(y_true),
        "positif_pct": round(100 * y_true.mean(), 2),
        "pr_auc": round(average_precision_score(y_true, prob), 4),
        "roc_auc": round(roc_auc_score(y_true, prob), 4),
        "threshold": round(threshold, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "brier": round(brier_score_loss(y_true, prob), 5),
        "logloss": round(log_loss(y_true, np.clip(prob, 1e-6, 1 - 1e-6)), 5),
    }


results = []
test_probs = {}
val_probs = {}
thresholds = {}

## Step 6: Baseline persistensi dan klimatologi

Baseline wajib. Model yang lebih rumit harus mengalahkan ini sebelum dianggap berguna.

In [ ]:
# Persistensi
val_probs["Persistence"] = val["ada_api"].astype(float).to_numpy()
test_probs["Persistence"] = test["ada_api"].astype(float).to_numpy()
thr = 0.5
thresholds["Persistence"] = thr
results.append(evaluate("Persistence", "validasi", val[TARGET], val_probs["Persistence"], thr))
results.append(evaluate("Persistence", "uji", test[TARGET], test_probs["Persistence"], thr))

# Klimatologi sel x bulan dari seluruh training, bukan sampel
clim = q(f"""
SELECT lat_c, lon_c, bulan, avg(y3) AS clim
FROM {P}
WHERE date <= DATE '{TRAIN_END}' AND y3 IS NOT NULL
GROUP BY 1, 2, 3
""")
global_rate = float(q(f"SELECT avg(y3) FROM {P} WHERE date <= DATE '{TRAIN_END}' AND y3 IS NOT NULL").iloc[0, 0])

val_clim = val.merge(clim, on=["lat_c", "lon_c", "bulan"], how="left")
test_clim = test.merge(clim, on=["lat_c", "lon_c", "bulan"], how="left")
val_clim["clim"] = val_clim["clim"].fillna(global_rate)
test_clim["clim"] = test_clim["clim"].fillna(global_rate)

val_probs["Climatology"] = val_clim["clim"].to_numpy()
test_probs["Climatology"] = test_clim["clim"].to_numpy()
thr = best_threshold(val[TARGET], val_probs["Climatology"])
thresholds["Climatology"] = thr
results.append(evaluate("Climatology", "validasi", val[TARGET], val_probs["Climatology"], thr))
results.append(evaluate("Climatology", "uji", test[TARGET], test_probs["Climatology"], thr))

display(pd.DataFrame(results).tail(4))

## Step 7: Siapkan matriks fitur

Tree model memakai nilai mentah dan boleh NaN. Regresi logistik dan MLP memakai `log1p` untuk fitur berekor panjang, median imputation, lalu standardisasi.

In [ ]:
def matrix_raw(df):
    return df[FEATURES].astype("float32")


def fit_dense_preprocessor(df):
    med = df[FEATURES].median(numeric_only=True)
    X = df[FEATURES].fillna(med).copy()
    for c in SKEWED:
        if c in X.columns:
            X[c] = np.log1p(X[c].clip(lower=0))
    scaler = StandardScaler()
    scaler.fit(X)
    return med, scaler


def matrix_dense(df, med, scaler):
    X = df[FEATURES].fillna(med).copy()
    for c in SKEWED:
        if c in X.columns:
            X[c] = np.log1p(X[c].clip(lower=0))
    return scaler.transform(X).astype("float32")


X_train_raw = matrix_raw(train)
X_val_raw = matrix_raw(val)
X_test_raw = matrix_raw(test)

# Random Forest tidak menerima NaN
med_raw = train[FEATURES].median(numeric_only=True)
X_train_imp = X_train_raw.fillna(med_raw)
X_val_imp = X_val_raw.fillna(med_raw)
X_test_imp = X_test_raw.fillna(med_raw)

med, scaler = fit_dense_preprocessor(train)
X_train_dense = matrix_dense(train, med, scaler)
X_val_dense = matrix_dense(val, med, scaler)
X_test_dense = matrix_dense(test, med, scaler)

y_train = train[TARGET].astype(int).to_numpy()
y_val = val[TARGET].astype(int).to_numpy()
y_test = test[TARGET].astype(int).to_numpy()

print("X_train_raw :", X_train_raw.shape)
print("X_train_dense:", X_train_dense.shape)
print("Positif train:", int(y_train.sum()))

## Step 8: Regresi logistik

In [ ]:
t0 = time.time()
logreg = LogisticRegression(
    max_iter=30 if SMOKE_TEST else 200,
    class_weight="balanced",
    solver="lbfgs",
    n_jobs=-1,
    random_state=SEED,
)
logreg.fit(X_train_dense, y_train)
val_probs["LogisticRegression"] = logreg.predict_proba(X_val_dense)[:, 1]
test_probs["LogisticRegression"] = logreg.predict_proba(X_test_dense)[:, 1]
thr = best_threshold(y_val, val_probs["LogisticRegression"])
thresholds["LogisticRegression"] = thr
results.append(evaluate("LogisticRegression", "validasi", y_val, val_probs["LogisticRegression"], thr))
results.append(evaluate("LogisticRegression", "uji", y_test, test_probs["LogisticRegression"], thr))
print(f"Selesai dalam {time.time() - t0:.1f} detik")
display(pd.DataFrame(results).tail(2))

## Step 9: Random Forest

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=20 if SMOKE_TEST else 150,
    max_depth=10 if SMOKE_TEST else 18,
    min_samples_leaf=50,
    max_samples=0.3,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=SEED,
)
rf.fit(X_train_imp, y_train)
val_probs["RandomForest"] = rf.predict_proba(X_val_imp)[:, 1]
test_probs["RandomForest"] = rf.predict_proba(X_test_imp)[:, 1]
thr = best_threshold(y_val, val_probs["RandomForest"])
thresholds["RandomForest"] = thr
results.append(evaluate("RandomForest", "validasi", y_val, val_probs["RandomForest"], thr))
results.append(evaluate("RandomForest", "uji", y_test, test_probs["RandomForest"], thr))
print(f"Selesai dalam {time.time() - t0:.1f} detik")
display(pd.DataFrame(results).tail(2))

# Bebaskan memori: XGBoost dan LightGBM menangani NaN sendiri.
del X_train_imp, X_val_imp, X_test_imp

## Step 10: LightGBM

Model utama. Early stopping memakai PR-AUC pada validasi.

In [ ]:
if not HAS_LGB:
    print("LightGBM tidak terpasang. Lewati.")
else:
    t0 = time.time()
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100 if SMOKE_TEST else 2000,
        learning_rate=0.05 if SMOKE_TEST else 0.03,
        num_leaves=31 if SMOKE_TEST else 63,
        min_child_samples=100,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary",
        is_unbalance=True,
        n_jobs=-1,
        random_state=SEED,
        verbosity=-1,
    )
    lgb_model.fit(
        X_train_raw, y_train,
        eval_set=[(X_val_raw, y_val)],
        eval_metric="average_precision",
        callbacks=[
            lgb.early_stopping(20 if SMOKE_TEST else 100, verbose=False),
            lgb.log_evaluation(50),
        ],
    )
    val_probs["LightGBM"] = lgb_model.predict_proba(X_val_raw)[:, 1]
    test_probs["LightGBM"] = lgb_model.predict_proba(X_test_raw)[:, 1]
    thr = best_threshold(y_val, val_probs["LightGBM"])
    thresholds["LightGBM"] = thr
    results.append(evaluate("LightGBM", "validasi", y_val, val_probs["LightGBM"], thr))
    results.append(evaluate("LightGBM", "uji", y_test, test_probs["LightGBM"], thr))
    print(f"Selesai dalam {time.time() - t0:.1f} detik")
    display(pd.DataFrame(results).tail(2))

## Step 11: XGBoost

In [ ]:
if not HAS_XGB:
    print("XGBoost tidak terpasang. Lewati.")
else:
    t0 = time.time()
    xgb_model = xgb.XGBClassifier(
        n_estimators=100 if SMOKE_TEST else 2000,
        learning_rate=0.05 if SMOKE_TEST else 0.03,
        max_depth=6 if SMOKE_TEST else 8,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        scale_pos_weight=2,
        n_jobs=-1,
        random_state=SEED,
        early_stopping_rounds=20 if SMOKE_TEST else 100,
    )
    xgb_model.fit(
        X_train_raw, y_train,
        eval_set=[(X_val_raw, y_val)],
        verbose=50,
    )
    val_probs["XGBoost"] = xgb_model.predict_proba(X_val_raw)[:, 1]
    test_probs["XGBoost"] = xgb_model.predict_proba(X_test_raw)[:, 1]
    thr = best_threshold(y_val, val_probs["XGBoost"])
    thresholds["XGBoost"] = thr
    results.append(evaluate("XGBoost", "validasi", y_val, val_probs["XGBoost"], thr))
    results.append(evaluate("XGBoost", "uji", y_test, test_probs["XGBoost"], thr))
    print(f"Selesai dalam {time.time() - t0:.1f} detik")
    display(pd.DataFrame(results).tail(2))

## Step 12: MLP neural network

PyTorch dipakai jika tersedia. Kalau tidak, notebook memakai `MLPClassifier` scikit-learn.

In [ ]:
if HAS_TORCH:
    class MLP(nn.Module):
        def __init__(self, n_features):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_features, 256),
                nn.BatchNorm1d(256),
                nn.ReLU(),
                nn.Dropout(0.20),
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(0.20),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 1),
            )

        def forward(self, x):
            return self.net(x).squeeze(-1)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(SEED)

    train_ds = TensorDataset(
        torch.from_numpy(X_train_dense),
        torch.from_numpy(y_train.astype(np.float32)),
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=8192,
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )

    n_pos = max(1, int(y_train.sum()))
    n_neg = len(y_train) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)

    mlp = MLP(X_train_dense.shape[1]).to(device)
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def predict_torch(model, X, batch_size=16384):
        model.eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                xb = torch.tensor(X[i:i + batch_size], dtype=torch.float32, device=device)
                out.append(torch.sigmoid(model(xb)).cpu().numpy())
        return np.concatenate(out)

    best_ap = -1.0
    best_state = None
    patience = 2 if SMOKE_TEST else 4
    wait = 0
    epochs = 2 if SMOKE_TEST else 20

    for epoch in range(1, epochs + 1):
        mlp.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = mlp(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)

        val_prob = predict_torch(mlp, X_val_dense)
        ap = average_precision_score(y_val, val_prob)
        print(f"Epoch {epoch:02d} | loss {total_loss / len(train_ds):.4f} | val PR-AUC {ap:.4f}")

        if ap > best_ap:
            best_ap = ap
            best_state = {k: v.detach().cpu().clone() for k, v in mlp.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break

    if best_state is not None:
        mlp.load_state_dict(best_state)

    val_probs["MLP"] = predict_torch(mlp, X_val_dense)
    test_probs["MLP"] = predict_torch(mlp, X_test_dense)

else:
    print("PyTorch tidak terpasang. Memakai MLPClassifier scikit-learn.")
    t0 = time.time()
    mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        batch_size=4096,
        learning_rate_init=1e-3,
        max_iter=5 if SMOKE_TEST else 40,
        early_stopping=True,
        n_iter_no_change=3,
        random_state=SEED,
        verbose=True,
    )
    mlp.fit(X_train_dense, y_train)
    val_probs["MLP"] = mlp.predict_proba(X_val_dense)[:, 1]
    test_probs["MLP"] = mlp.predict_proba(X_test_dense)[:, 1]
    print(f"Selesai dalam {time.time() - t0:.1f} detik")

thr = best_threshold(y_val, val_probs["MLP"])
thresholds["MLP"] = thr
results.append(evaluate("MLP", "validasi", y_val, val_probs["MLP"], thr))
results.append(evaluate("MLP", "uji", y_test, test_probs["MLP"], thr))
display(pd.DataFrame(results).tail(2))

## Step 13: Perbandingan model

In [ ]:
res = pd.DataFrame(results)
res = res.sort_values(["split", "pr_auc"], ascending=[True, False])
display(res)

pivot = res.pivot(index="model", columns="split", values="pr_auc").sort_values("uji", ascending=False)
display(pivot)

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_title("Perbandingan PR-AUC pada Validasi dan Uji")
ax.set_ylabel("PR-AUC")
ax.set_ylim(0, max(0.35, pivot.max().max() * 1.15))
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "perbandingan_pr_auc.png", dpi=150)
plt.show()

## Step 14: Evaluasi per tahun dan per musim

Model terbaik dipilih berdasarkan PR-AUC validasi, bukan uji.

In [ ]:
best_model_name = res[res["split"] == "validasi"].sort_values("pr_auc", ascending=False).iloc[0]["model"]
print("Model terbaik menurut validasi:", best_model_name)

best_prob = test_probs[best_model_name]
best_thr = thresholds[best_model_name]

test_eval = test[["date", "bulan", TARGET]].copy()
test_eval["prob"] = best_prob
test_eval["year"] = pd.to_datetime(test_eval["date"]).dt.year
test_eval["season"] = np.where(test_eval["bulan"].isin([7, 8, 9, 10, 11]), "Jul-Nov", "Des-Jun")

per_year = []
for y, g in test_eval.groupby("year"):
    r = evaluate(best_model_name, f"uji-{y}", g[TARGET], g["prob"], best_thr)
    per_year.append(r)
display(pd.DataFrame(per_year))

per_season = []
for s, g in test_eval.groupby("season"):
    r = evaluate(best_model_name, f"uji-{s}", g[TARGET], g["prob"], best_thr)
    per_season.append(r)
display(pd.DataFrame(per_season))

## Step 15: Feature importance LightGBM

In [ ]:
if HAS_LGB and "lgb_model" in globals():
    imp = pd.DataFrame({
        "fitur": FEATURES,
        "gain": lgb_model.booster_.feature_importance(importance_type="gain"),
        "split": lgb_model.booster_.feature_importance(importance_type="split"),
    }).sort_values("gain", ascending=False)

    display(imp.head(20))

    fig, ax = plt.subplots(figsize=(9, 7))
    top = imp.head(20).sort_values("gain")
    ax.barh(top["fitur"], top["gain"], color="darkorange")
    ax.set_title("LightGBM Feature Importance (gain)")
    ax.set_xlabel("Gain")
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "feature_importance_lgb.png", dpi=150)
    plt.show()
else:
    print("LightGBM tidak tersedia, lewati feature importance.")

## Step 16: Simpan artefak

File yang disimpan:
- `model_metrics.csv`
- `model_thresholds.json`
- model terbaik (joblib atau state_dict)
- `feature_list.json`

Model biner ini belum dikalibrasi. Kalibrasi isotonik atau Platt scaling perlu dilakukan sebelum probabilitas dipakai sebagai tingkat risiko.

In [ ]:
res.to_csv(METRICS_PATH, index=False)
with open(THRESHOLD_PATH, "w") as f:
    json.dump(thresholds, f, indent=2)
with open(ARTIFACT_DIR / "feature_list.json", "w") as f:
    json.dump(FEATURES, f, indent=2)

saved = []

if best_model_name == "LightGBM" and HAS_LGB and "lgb_model" in globals():
    lgb_model.booster_.save_model(str(MODEL_DIR / "lightgbm.txt"))
    saved.append("lightgbm.txt")
elif best_model_name == "XGBoost" and HAS_XGB and "xgb_model" in globals():
    joblib.dump(xgb_model, MODEL_DIR / "xgboost.joblib")
    saved.append("xgboost.joblib")
elif best_model_name == "RandomForest" and "rf" in globals():
    joblib.dump(rf, MODEL_DIR / "random_forest.joblib")
    saved.append("random_forest.joblib")
elif best_model_name == "LogisticRegression" and "logreg" in globals():
    joblib.dump({"model": logreg, "median": med, "scaler": scaler}, MODEL_DIR / "logistic_regression.joblib")
    saved.append("logistic_regression.joblib")
elif best_model_name == "MLP":
    if HAS_TORCH and "mlp" in globals():
        torch.save(mlp.state_dict(), MODEL_DIR / "mlp.pt")
        joblib.dump({"median": med, "scaler": scaler}, MODEL_DIR / "mlp_preprocessor.joblib")
        saved.append("mlp.pt")
    elif "mlp" in globals():
        joblib.dump({"model": mlp, "median": med, "scaler": scaler}, MODEL_DIR / "mlp_sklearn.joblib")
        saved.append("mlp_sklearn.joblib")

print("Metrik  :", METRICS_PATH)
print("Ambang  :", THRESHOLD_PATH)
print("Model   :", saved)
print()
print("Catatan: baseline terbaik adalah pembanding minimum. Kalau model iklim murni")
print("tidak mengalahkan persistensi, tambahkan fitur lag hotspot atau tetangga.")

## Step 17: Cara membaca hasil

1. Bandingkan **PR-AUC uji**, bukan akurasi. Dengan positif sekitar 8%, akurasi 92% bisa dicapai hanya dengan menebak semua negatif.
2. Bandingkan setiap model dengan **persistensi**. Kalau kalah, model belum menambah nilai.
3. Lihat **recall pada ambang validasi**. Untuk peringatan dini, recall lebih penting daripada precision, tetapi terlalu banyak positif palsu membuat sistem tidak dipercaya.
4. Cek **per tahun** dan **Jul–Nov**. Model yang bagus secara agregat bisa gagal total pada tahun El Niño.
5. Cek **feature importance**. Kalau `lat_c`/`lon_c` mendominasi, model menghafal lokasi, bukan belajar cuaca.
6. Kalibrasi probabilitas sebelum memetakan ke kelas risiko Low/Moderate/High/Extreme.

## Batasan

- Sampel negatif 2× positif mempercepat training, tetapi mengubah kalibrasi.
- Label `ada_api=0` berarti tidak terdeteksi, bukan pasti tidak ada api.
- Cuaca yang dipakai adalah observasi. Model operasional membutuhkan prakiraan cuaca.
- Belum ada fitur tetangga spasial. Penyebaran api belum tertangkap.
- Belum ada validasi rolling-origin. Untuk laporan akhir, ulangi evaluasi per tahun.
- 441 sel hotspot tidak masuk panel karena ERA5 tidak punya grid di sana.

## Langkah lanjutan

1. Kalibrasi isotonik pada validasi.
2. Tambah fitur tetangga: rata-rata `fwi` dan jumlah hotspot 8 sel sekitar, lag 1–3 hari.
3. Uji rolling-origin 2019–2026.
4. Uji ablation: iklim saja, hotspot lag saja, gabungan.
5. Latih model per horizon: `lead1`, `lead2`, `lead3` terpisah.
6. Tambah fitur statis: gambut, tutupan lahan, elevasi, jarak jalan.